In [4]:
import polars as pl

print("--- Olympic Medal Intelligence Engine Starting ---")

# 1. Loading data from your own local file
df = pl.read_csv("athlete_events.csv", null_values="NA", infer_schema_length=None)

# 2. Filtering only medal winners
medal_df = df.filter(pl.col("Medal").is_not_null())

# 3. Counting Gold, Silver, Bronze for each country
country_tally = medal_df.group_by("NOC").agg([
    pl.col("Medal").filter(pl.col("Medal") == "Gold").count().alias("Gold"),
    pl.col("Medal").filter(pl.col("Medal") == "Silver").count().alias("Silver"),
    pl.col("Medal").filter(pl.col("Medal") == "Bronze").count().alias("Bronze")
])

# 4. Applying weighted performance score (Gold=3, Silver=2, Bronze=1)
ranked_df = country_tally.with_columns(
    ((pl.col("Gold") * 3) + (pl.col("Silver") * 2) + (pl.col("Bronze") * 1)).alias("Performance_Score")
)

# 5. Sorting to get the top ranks
sorted_df = ranked_df.sort("Performance_Score", descending=True)
final_ranked_df = sorted_df.with_row_index(name="Rank", offset=1)

# 6. Displaying Top 10 Nations
print("\n🏆 Top 10 Olympic Nations Leaderboard:")
final_ranked_df.head(10)

--- Olympic Medal Intelligence Engine Starting ---

🏆 Top 10 Olympic Nations Leaderboard:


Rank,NOC,Gold,Silver,Bronze,Performance_Score
u32,str,u32,u32,u32,u32
1,"""USA""",2638,1641,1358,12554
2,"""URS""",1082,732,689,5399
3,"""GER""",745,674,746,4329
4,"""GBR""",678,739,651,4163
5,"""FRA""",501,610,666,3389
6,"""ITA""",575,531,531,3318
7,"""SWE""",479,522,535,3016
8,"""CAN""",463,438,451,2716
9,"""AUS""",348,455,517,2471
